# 表格文件清洗 · CSV / TSV / XLSX 统一入口

**样本**：
- `game-items-zh.*` — 干净的 8 行 × 5 列中文表格（三种格式内容相同，用于对照验证）
- `stanley-cups.*` — 脏表头（第一行不是真正的表头），用于边界测试

**方法**：
- 按扩展名路由：`.csv` → `csv` 模块；`.tsv` → `csv` 模块（分隔符换成 Tab）；`.xlsx` → `openpyxl`
- 读成二维列表（行 × 列）→ 每行用 ` | ` 拼成文本 → 生成一个 `Table` 类型的 Element
- xlsx 额外处理：逐 sheet 遍历，每个非空 sheet 生成一个 Element；读显示值而非公式
- 脏表头兜底：若第一行列数和第二行不同，把第一行单独当标题文本处理
- 分块：纯表格文件没有正文段落，整表直接作为一块（小表）

**依赖**：`pip install openpyxl`（csv / tsv 只需标准库）

In [ ]:
import csv
import re
from dataclasses import dataclass, field
from pathlib import Path

import openpyxl


def clean_text(text: str) -> str:
    '''合并连续空白，去首尾空白。'''
    return re.sub(r'\s+', ' ', str(text)).strip()


@dataclass
class Element:
    '''解析出的一个表格元素。'''
    index:    int
    category: str   # 只有 Table 和 Title 两种
    text:     str   # 清洗后的文字；Table 格式为「列1 | 列2 | ...」逐行
    sheet:    str = ''  # 仅 xlsx 有值，记录来自哪个 sheet

In [ ]:
def _rows_to_text(rows: list[list[str]]) -> str:
    '''把二维列表转为纯文本，每行用 " | " 分隔列。空行跳过。'''
    lines = []
    for row in rows:
        cells = [clean_text(c) for c in row]
        if any(cells):  # 跳过全空行
            lines.append(' | '.join(cells))
    return '\n'.join(lines)


def _is_dirty_header(rows: list[list[str]]) -> bool:
    '''
    判断表格是否有「脏表头」：
    - 第一行非空列数 < 第二行非空列数，说明第一行更像标题而非真正的表头
    - stanley-cups.csv 第一行 "Stanley Cups,," 就是这种情况
    '''
    if len(rows) < 2:
        return False
    first_nonempty  = sum(1 for c in rows[0] if clean_text(c))
    second_nonempty = sum(1 for c in rows[1] if clean_text(c))
    return first_nonempty < second_nonempty


def _read_csv_rows(file_path: Path, delimiter: str) -> list[list[str]]:
    '''读 csv / tsv，返回二维列表。自动处理 UTF-8 BOM（Excel 导出常见）。'''
    with open(file_path, encoding='utf-8-sig', newline='') as f:
        return list(csv.reader(f, delimiter=delimiter))


def _read_xlsx_sheets(file_path: Path) -> dict[str, list[list[str]]]:
    '''
    读 xlsx，返回 {sheet名: 二维列表}。
    - data_only=True：读单元格显示值，不读公式字符串
    - 跳过只有 1 行或全空的 sheet
    '''
    wb = openpyxl.load_workbook(file_path, data_only=True)
    result = {}
    for ws in wb.worksheets:
        rows = [[str(cell.value) if cell.value is not None else '' for cell in row]
                for row in ws.iter_rows()]
        # 跳过空 sheet
        if not any(any(clean_text(c) for c in row) for row in rows):
            continue
        result[ws.title] = rows
    return result


def extract(file_path: Path) -> list[Element]:
    '''
    统一入口：按扩展名路由，返回 Element 列表。

    .csv  → 逗号分隔
    .tsv  → Tab 分隔
    .xlsx → 逐 sheet 读，每个 sheet 一个 Element

    脏表头处理：若第一行列数 < 第二行，把第一行当 Title Element，
    剩余行当 Table Element。
    '''
    suffix = file_path.suffix.lower()
    elements: list[Element] = []

    if suffix == '.csv':
        sheets = {'': _read_csv_rows(file_path, delimiter=',')}
    elif suffix == '.tsv':
        sheets = {'': _read_csv_rows(file_path, delimiter='\t')}
    elif suffix == '.xlsx':
        sheets = _read_xlsx_sheets(file_path)
    else:
        raise ValueError(f'不支持的格式：{suffix}（仅支持 .csv .tsv .xlsx）')

    for sheet_name, rows in sheets.items():
        if not rows:
            continue

        # 脏表头：第一行单独当 Title
        if _is_dirty_header(rows):
            title_text = clean_text(' '.join(c for c in rows[0] if clean_text(c)))
            elements.append(Element(
                index=len(elements),
                category='Title',
                text=title_text,
                sheet=sheet_name,
            ))
            rows = rows[1:]  # 剩余行才是真正的表格数据

        table_text = _rows_to_text(rows)
        if table_text.strip():
            elements.append(Element(
                index=len(elements),
                category='Table',
                text=table_text,
                sheet=sheet_name,
            ))

    return elements

In [ ]:
# 跑三种格式的干净样本，内容应该完全一致
base = Path('.')
files = [
    base / 'game-items-zh.csv',
    base / 'game-items-zh.tsv',
    base / 'game-items-zh.xlsx',
]

for fp in files:
    els = extract(fp)
    print(f'文件: {fp.name}  → {len(els)} 个元素')
    for e in els:
        tag = '📋' if e.category == 'Table' else '🔖'
        sheet_info = f' [sheet: {e.sheet}]' if e.sheet else ''
        print(f'  {tag} ({e.category}{sheet_info})')
        print(f'    {e.text[:120]}')
    print()

In [ ]:
# 边界测试：脏表头样本
dirty_files = [
    base / 'stanley-cups.csv',
    base / 'stanley-cups.tsv',
    base / 'stanley-cups.xlsx',
]

for fp in dirty_files:
    els = extract(fp)
    print(f'文件: {fp.name}  → {len(els)} 个元素')
    for e in els:
        tag = '📋' if e.category == 'Table' else '🔖'
        print(f'  {tag} ({e.category})')
        for line in e.text.split('\n'):
            print(f'    {line}')
    print()

## RAG 分块：整表直接成块

纯表格文件和 HTML / DOCX 不同——**整个文件就是一张表，没有正文段落**，不需要标题 flush 和滑动窗口。

分块规则极简：

| 元素类型 | 处理方式 |
|----------|----------|
| Title（脏表头） | 写进 `section` 字段，不单独成块 |
| Table | 直接成块（小表整表一块） |

| 参数 | 说明 |
|------|------|
| `source_file` | 来源文件名，写进元数据 |
| `min_chunk_size` | 过短的块直接丢弃，默认 5 |

In [ ]:
@dataclass
class RagChunk:
    '''RAG 用的一块数据，带元信息方便检索。'''
    text:        str
    source_file: str
    section:     str        # 脏表头文字；干净表格为文件名
    sheet:       str        # xlsx 的 sheet 名；csv/tsv 为空字符串
    chunk_index: int
    char_count:  int = field(init=False)

    def __post_init__(self):
        self.char_count = len(self.text)


def chunk_elements(
    elements: list[Element],
    source_file: str,
    min_chunk_size: int = 5,
) -> list[RagChunk]:
    '''
    把 Element 列表转成 RagChunk 列表。

    - Title → 不成块，只更新 current_section
    - Table → 直接成块（整表一块，小表够用）
    '''
    chunks: list[RagChunk] = []
    current_section = source_file  # 默认用文件名当 section

    for el in elements:
        if el.category == 'Title':
            current_section = el.text  # 脏表头文字当 section
        elif el.category == 'Table':
            if len(el.text.strip()) >= min_chunk_size:
                chunks.append(RagChunk(
                    text=el.text.strip(),
                    source_file=source_file,
                    section=current_section,
                    sheet=el.sheet,
                    chunk_index=len(chunks),
                ))

    return chunks

In [ ]:
# 所有样本跑一遍，打印最终 RagChunk
all_files = [
    base / 'game-items-zh.csv',
    base / 'game-items-zh.tsv',
    base / 'game-items-zh.xlsx',
    base / 'stanley-cups.csv',
    base / 'stanley-cups.tsv',
    base / 'stanley-cups.xlsx',
]

for fp in all_files:
    els    = extract(fp)
    chunks = chunk_elements(els, source_file=fp.name)
    print(f'📁 {fp.name}  → {len(chunks)} 块')
    for c in chunks:
        sheet_info = f' sheet={c.sheet}' if c.sheet else ''
        print(f'  [{c.chunk_index}] section={c.section}{sheet_info}  len={c.char_count}')
        for line in c.text.split('\n'):
            print(f'      {line}')
    print()